# Phase 1 — Agent Interface Design

**Developer 2 | Detection / Agents workstream**

This notebook is the Phase 1 deliverable for the Detection workstream. Its purpose
is to document and validate the `BaseAgent` contract every Neural Sentinel agent must
satisfy, explore the canonical data contracts, build a minimal `StubAgent`, confirm
config and Nepal-context integration, verify serialisability, and preview the full
eight-agent roster planned for later phases.

| # | Section | Purpose |
|---|---|---|
| 1 | Environment Setup | Path bootstrap, packages, seeds, logging, GPU check |
| 2 | Canonical Data Contracts | `Transaction`, `Account`, `AlertScore` Pydantic models |
| 3 | BaseAgent Contract Deep-Dive | Every method, class var, and design rule |
| 4 | StubAgent — Minimal Concrete Implementation | fit / predict / explain cycle |
| 5 | Config & Nepal-Context Integration | How agents consume thresholds and domain constants |
| 6 | Agent Contract Compliance Checklist | Verify each rule from AGENTS.md §8.1 |
| 7 | Serialisation | pickle and joblib round-trip |
| 8 | Agent Registry Preview | Blueprint for the full multi-agent roster |
| 9 | Inline Test-Suite Dry Run | Execute Phase 1 tests without leaving the notebook |
| 10 | Phase 1 Summary | Key decisions, open questions, handoff to Phase 2 |


---
## 1. Environment Setup

Follows AGENTS.md §10.4: detect GPU, install missing packages, set seeds, configure logging.

In [ ]:
# 1a. Path bootstrap — works on local dev and Kaggle
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path('/kaggle/working/neural-sentinel')
assert (PROJECT_ROOT / 'src').exists(), (
    f'src/ not found under PROJECT_ROOT={PROJECT_ROOT}. '
    'Update PROJECT_ROOT manually or mount the Kaggle dataset.'
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT resolved to: {PROJECT_ROOT}')


In [ ]:
# 1b. Install missing packages (no-ops when already present)
import importlib
import subprocess

_REQUIRED = {
    'pydantic':  'pydantic==2.12.5',
    'catboost':  'catboost==1.2.10',
    'sklearn':   'scikit-learn==1.6.1',
    'joblib':    'joblib==1.4.2',
}
for _mod, _spec in _REQUIRED.items():
    if importlib.util.find_spec(_mod) is None:
        print(f'Installing {_spec} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _spec])
    else:
        print(f'{_mod}: OK')


In [ ]:
# 1c. Reproducibility — set all random seeds
import random
import numpy as np

RANDOM_SEED = 42 
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    print(f'torch seed: {RANDOM_SEED}')
except ImportError:
    print('PyTorch not available — CPU-only run, torch seed skipped')

print(f'numpy/random seed: {RANDOM_SEED}')


In [ ]:
# 1d. Logging — structured format used throughout the project
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('phase1_notebook')
logger.info('Logging ready — Phase 1 Agent Interfaces')


In [ ]:
# 1e. GPU / accelerator detection (AGENTS.md §7.3)
# Phase 1 does not require GPU; pattern is established here for later phases.
import os

DEVICE = 'cpu'
try:
    import torch as _torch
    if _torch.cuda.is_available():
        DEVICE = 'cuda'
        print(f'CUDA GPU: {_torch.cuda.get_device_name(0)}')
    elif hasattr(_torch.backends, 'mps') and _torch.backends.mps.is_available():
        DEVICE = 'mps'
        print('Apple MPS (Metal) GPU available')
    else:
        print('No GPU — CPU only (expected for Phase 1)')
except ImportError:
    print('PyTorch absent — CPU only')

if os.environ.get('TPU_NAME'):
    DEVICE = 'tpu'
    print(f"Kaggle TPU: {os.environ['TPU_NAME']}")

print(f'Active device: {DEVICE}')


---
## 2. Canonical Data Contracts

Three Pydantic v2 models in `src/data_contracts.py` form the **single source of truth**
for all data flowing between agents (AGENTS.md §5).

| Model | Table | Rows produced by |
|---|---|---|
| `Transaction` | `transactions` | Data pipeline (Dev 1) |
| `Account` | `accounts` | Data pipeline (Dev 1) |
| `AlertScore` | `alert_scores` | Every agent (Dev 2) |

Agents **read** `Transaction` and `Account` rows and **write** `AlertScore` rows.
Every field is documented in the model's docstring; validation errors are raised
at parse time so invalid data never reaches model code.

In [ ]:
from src.data_contracts import Transaction, Account, AlertScore
import datetime, pprint

# --- Inspect field names and types ---
print('=== Transaction fields ===')
for name, field in Transaction.model_fields.items():
    print(f'  {name:<30} {str(field.annotation)}')


In [ ]:
print('\n=== Account fields ===')
for name, field in Account.model_fields.items():
    print(f'  {name:<30} {str(field.annotation)}')

print('\n=== AlertScore fields ===')
for name, field in AlertScore.model_fields.items():
    print(f'  {name:<30} {str(field.annotation)}')


In [ ]:
# --- Build a valid Transaction to confirm schema acceptance ---
txn = Transaction(
    transaction_id='txn-demo-001',
    transaction_date=datetime.date(2025, 3, 15),
    transaction_time=datetime.time(14, 32, 10),
    sender_account_id='ACC0000000000001',
    receiver_account_id='ACC0000000000002',
    transaction_type='remittance_inbound',
    amount_npr=450_000.0,
    original_currency='QAR',
    exchange_rate=34.5,
    channel='mobile_banking',
    sender_country='Qatar',
    receiver_country='Nepal',
    is_cross_border=1,
    remittance_corridor='Qatar->Nepal',
    merchant_category=None,
    device_type='mobile',
    ip_address='192.168.1.10',
    ip_country='Qatar',
    ip_is_vpn=0,
    is_fraud=0,
    fraud_type=None,
    aml_risk_indicator=0,
)
print('Transaction validated successfully:')
pprint.pprint(txn.model_dump())


In [ ]:
# --- Demonstrate validation error: cross-border without corridor ---
from pydantic import ValidationError

try:
    bad = Transaction(
        transaction_id='txn-bad',
        transaction_date=datetime.date(2025, 1, 1),
        transaction_time=datetime.time(0, 0, 0),
        sender_account_id='ACC001',
        receiver_account_id='ACC002',
        transaction_type='transfer',
        amount_npr=100.0,
        original_currency='USD',
        exchange_rate=120.0,
        channel='online_banking',
        sender_country='USA',
        receiver_country='Nepal',
        is_cross_border=1,          # cross-border ...
        remittance_corridor=None,   # ... but no corridor  <-- invalid
        device_type='desktop',
        ip_address='1.2.3.4',
        ip_is_vpn=0,
        is_fraud=0,
        aml_risk_indicator=0,
    )
except ValidationError as exc:
    print('Caught expected ValidationError:')
    print(exc)


In [ ]:
# --- Build a valid AlertScore (output contract) ---
import datetime as dt

alert = AlertScore(
    transaction_id='txn-demo-001',
    agent_name='geo_risk',
    risk_score=0.82,
    alert_flag=1,
    reason_code='HIGH_RISK_CORRIDOR',
    explanation=(
        'Transaction flagged: NPR 450,000 remittance inbound via Qatar->Nepal corridor '
        '(high-volume, medium-risk) at 14:32 via mobile banking.'
    ),
    timestamp=dt.datetime.now(dt.timezone.utc),
)
print('AlertScore validated:')
pprint.pprint(alert.model_dump())


---
## 3. BaseAgent Contract Deep-Dive

`BaseAgent` (AGENTS.md §8.1) is the abstract root class for every detection agent.
The rules every subclass must follow:

1. Inherit from `BaseAgent`.
2. Accept a `config` dict/Pydantic model and a `logger` at `__init__`.
3. Implement `fit(data) -> self`.
4. Implement `predict(data) -> pd.DataFrame` returning at minimum: `transaction_id`,
   `risk_score`, `alert_flag`, `reason_code`.
5. Implement `explain(transaction_id) -> str`.
6. Be serialisable (pickle / joblib).
7. Log all decisions via the structured logger — never `print()`.
8. Never mutate input data.
9. Handle missing features gracefully — log a warning, impute or skip, never crash.

The subsections below inspect each class variable, method, and helper.

In [ ]:
import inspect
from src.agents.base_agent import BaseAgent

# Class variables every subclass inherits
print('agent_name (ClassVar):         ', BaseAgent.agent_name)
print('default_alert_threshold:       ', BaseAgent.default_alert_threshold)
print('prediction_columns (tuple):    ')
for col in BaseAgent.prediction_columns:
    print(f'  - {col}')


In [ ]:
# Abstract methods — subclasses MUST override these
abstract_methods = {
    name
    for name, val in inspect.getmembers(BaseAgent)
    if getattr(val, '__isabstractmethod__', False)
}
print('Abstract methods (must be implemented by every agent):')
for m in sorted(abstract_methods):
    sig = inspect.signature(getattr(BaseAgent, m))
    print(f'  {m}{sig}')


In [ ]:
# Concrete helpers provided by BaseAgent
_helpers = [
    'build_predictions',
    'empty_predictions',
    'require_columns',
    '_resolve_alert_threshold',
    '_config_to_dict',
]
for name in _helpers:
    fn = getattr(BaseAgent, name)
    print(f'{name}{inspect.signature(fn)}')
    doc = (inspect.getdoc(fn) or '').split('\n')[0]
    print(f'  → {doc}\n')


### 3.1 `build_predictions` — canonical output builder

`build_predictions(data, risk_scores, reason_code, explanation)` is the **only** way
an agent should materialise its output DataFrame. It:

- Copies `transaction_id` from input (never mutates).
- Clips scores to `[0, 1]` and logs a warning when clipping is required.
- Sets `alert_flag = 1` when `risk_score >= alert_threshold`.
- Appends `agent_name` and a UTC `timestamp`.
- Returns exactly `BaseAgent.prediction_columns` — no extra columns.

In [ ]:
import pandas as pd
import numpy as np

# Demonstrate build_predictions via a temporary concrete subclass
class _DemoAgent(BaseAgent):
    agent_name = 'demo'
    def fit(self, data): self.is_fitted = True; return self
    def predict(self, data): return self.build_predictions(data, [0.9])
    def explain(self, tid): return f'demo explanation for {tid}'

sample = pd.DataFrame({'transaction_id': ['txn-X'], 'amount_npr': [200_000.0]})
out = _DemoAgent(config={'demo_alert_threshold': 0.6}).fit(sample).predict(sample)
print('Output columns:', list(out.columns))
print('Output dtypes:')
print(out.dtypes)
print('\nRow:')
print(out.to_string(index=False))


In [ ]:
# Score clipping: out-of-range values are clipped and a warning is emitted
out_clipped = _DemoAgent().fit(sample).build_predictions(sample, [1.8], reason_code='CLIPPING_TEST')
print(f'Input score 1.8 → clipped risk_score: {out_clipped.risk_score.iloc[0]}')


### 3.2 `empty_predictions` — schema-safe empty output

When an agent cannot process input (e.g. `transaction_id` column missing), it must
return an empty DataFrame with the correct schema rather than raising. This ensures
downstream joins don't crash.

In [ ]:
empty = BaseAgent.empty_predictions()
print('empty_predictions() columns:', list(empty.columns))
print('empty_predictions() dtypes:')
print(empty.dtypes)
print('Is empty:', empty.empty)


### 3.3 `require_columns` — graceful missing-feature handling

Agents call `self.require_columns(data, ('col_a', 'col_b'))` at the top of `predict()`.
If any column is absent, a warning is logged and `False` is returned so the caller
can return `empty_predictions()` — it never raises.

In [ ]:
agent = _DemoAgent()
df_good = pd.DataFrame({'transaction_id': ['t1'], 'amount_npr': [50.0]})
df_bad  = pd.DataFrame({'amount_npr': [50.0]})  # transaction_id missing

print('Has transaction_id:', agent.require_columns(df_good, ('transaction_id',)))
print('Missing transaction_id:', agent.require_columns(df_bad, ('transaction_id',)))
# The warning is emitted to the logger — check the log output above


### 3.4 Alert threshold resolution order

`_resolve_alert_threshold` checks config keys in this priority order:

1. `<agent_name>_alert_threshold` — agent-specific override (e.g. `velocity_alert_threshold`)
2. `<agent_name_without_'_agent'>_alert_threshold` — shorthand form
3. `alert_threshold` — generic fallback
4. `BaseAgent.default_alert_threshold` = 0.5 — hard default

In [ ]:
# Resolution order demo
cases = [
    ('agent-specific key',  {'velocity_alert_threshold': 0.7}),
    ('generic key',         {'alert_threshold': 0.3}),
    ('empty config',        {}),
]

class _VAgent(BaseAgent):
    agent_name = 'velocity'
    def fit(self, d): return self
    def predict(self, d): return self.empty_predictions()
    def explain(self, t): return ''

for label, cfg in cases:
    a = _VAgent(config=cfg)
    print(f'{label:<30} → alert_threshold = {a.alert_threshold}')


---
## 4. StubAgent — Minimal Concrete Implementation

A `StubAgent` is a fully compliant agent that does the minimum necessary to satisfy the
contract. It is used in the test suite and serves as the canonical template every new
agent author should follow before adding real model logic.

The implementation requirements are deliberately minimal:
- `fit` sets `self.is_fitted = True` and returns `self`.
- `predict` validates input, then calls `build_predictions` with a fixed score.
- `explain` returns a simple f-string.

Everything else — logging, threshold resolution, serialisation, output schema — is
inherited from `BaseAgent`.

In [ ]:
from src.agents.base_agent import BaseAgent
import pandas as pd


class StubAgent(BaseAgent):
    """Minimal BaseAgent implementation used for interface validation.

    This is the exact class used in tests/test_agents.py.  It does not train
    a model — it always returns a fixed risk score of 0.2.
    """

    agent_name = 'stub'

    def fit(self, data: pd.DataFrame) -> 'StubAgent':
        """Mark the agent as fitted (no training required)."""
        self.is_fitted = True
        self.logger.info('StubAgent fitted on %d rows', len(data))
        return self

    def predict(self, data: pd.DataFrame) -> pd.DataFrame:
        """Return a constant risk score of 0.2 for every input row."""
        if not self.require_columns(data, ('transaction_id',)):
            return self.empty_predictions()
        return self.build_predictions(
            data,
            risk_scores=[0.2] * len(data),
            reason_code='STUB_SCORE',
            explanation='Stub agent: constant baseline score of 0.2.',
        )

    def explain(self, transaction_id: str) -> str:
        """Return a template explanation."""
        return (
            f'Transaction {transaction_id}: stub baseline score of 0.2 — '
            'no real model logic is applied in the stub implementation.'
        )


print('StubAgent defined successfully')


In [ ]:
# ── Fit / predict cycle ───────────────────────────────────────────────────
sample_data = pd.DataFrame({
    'transaction_id': ['txn-001', 'txn-002', 'txn-003'],
    'amount_npr': [50_000.0, 1_200_000.0, 8_500.0],
    'is_fraud': [0, 1, 0],
})

stub = StubAgent(config={'stub_alert_threshold': 0.15})
stub.fit(sample_data)
predictions = stub.predict(sample_data)

print('Output shape:', predictions.shape)
print('Columns match prediction_columns:', list(predictions.columns) == list(BaseAgent.prediction_columns))
print()
print(predictions.to_string(index=False))


In [ ]:
# ── explain() ──────────────────────────────────────────────────────────────
for tid in sample_data['transaction_id']:
    print(stub.explain(tid))


In [ ]:
# ── Missing-input graceful handling ────────────────────────────────────────
bad_input = pd.DataFrame({'amount_npr': [99.0, 200.0]})  # no transaction_id
result_bad = stub.predict(bad_input)
print('Empty result on bad input:', result_bad.empty)
print('Schema preserved:', list(result_bad.columns) == list(BaseAgent.prediction_columns))


In [ ]:
# ── Input immutability check ────────────────────────────────────────────────
original_copy = sample_data.copy(deep=True)
_ = stub.predict(sample_data)
mutated = not sample_data.equals(original_copy)
print('Input mutated by predict():', mutated)  # must be False


---
## 5. Config & Nepal-Context Integration

AGENTS.md §10.7 mandates that **all magic numbers and thresholds live in `config.py`**.
No hardcoded values may appear in agent code.  Agents receive the config object at
construction time and look up their specific threshold key.

In [ ]:
from src.utils.config import get_config
from src.utils.nepal_context import (
    REMITTANCE_CORRIDORS,
    CORRIDOR_RISK_SCORES,
    NRB_CASH_REPORTING_THRESHOLD_NPR,
    CHANNEL_MIX,
    NEPALI_CITIES,
    EXCHANGE_RATE_RANGES,
    CORRIDOR_CURRENCIES,
)

cfg = get_config()
print('=== Per-agent alert thresholds ===')
threshold_fields = [
    'velocity_alert_threshold',
    'geo_risk_alert_threshold',
    'kyc_aml_alert_threshold',
    'behaviour_alert_threshold',
    'graph_alert_threshold',
    'meta_learner_alert_threshold',
]
for f in threshold_fields:
    print(f'  {f:<40} {getattr(cfg, f)}')


In [ ]:
print('=== NRB regulatory thresholds ===')
print(f'  NRB cash reporting threshold (NPR): {cfg.nrb_cash_reporting_threshold_npr:,.0f}')
print(f'  Structuring detection window:       NPR {cfg.structuring_min_npr:,.0f} – {cfg.structuring_max_npr:,.0f}')

print('\n=== KYC/AML rule weights ===')
weight_fields = [
    'weight_pep_flag',
    'weight_sanctions_match',
    'weight_kyc_unverified',
    'weight_new_account',
    'weight_high_risk_grade',
    'weight_structuring_pattern',
    'weight_layering_pattern',
]
for f in weight_fields:
    print(f'  {f:<40} {getattr(cfg, f)}')


In [ ]:
print('=== Nepal remittance corridors ===')
for corridor, risk_tier in REMITTANCE_CORRIDORS.items():
    score = CORRIDOR_RISK_SCORES[risk_tier]
    currency = CORRIDOR_CURRENCIES.get(corridor, 'N/A')
    print(f'  {corridor:<30} risk={risk_tier:<8} score={score:.2f}  currency={currency}')


In [ ]:
print('=== NRB cash reporting threshold ===')
print(f'  NPR {NRB_CASH_REPORTING_THRESHOLD_NPR:,.0f}')

print('\n=== Channel mix (approximate Nepali banking) ===')
for channel, weight in sorted(CHANNEL_MIX.items(), key=lambda x: -x[1]):
    print(f'  {channel:<20} {weight*100:.1f}%')

print('\n=== NPR exchange rate ranges ===')
for ccy, (lo, hi) in EXCHANGE_RATE_RANGES.items():
    print(f'  {ccy}  NPR {lo:.2f} – {hi:.2f}')


In [ ]:
# How an agent consumes config — concrete example with StubAgent
import pprint

cfg_dict = cfg.model_dump()
stub_from_cfg = StubAgent(config=cfg_dict)
print('StubAgent initialised from Config singleton')
print(f'  alert_threshold = {stub_from_cfg.alert_threshold}')
print(f'  config keys (sample):')
for k in list(cfg_dict)[:8]:
    print(f'    {k}: {cfg_dict[k]}')


---
## 6. Agent Contract Compliance Checklist

AGENTS.md §8.1 lists nine rules every agent must follow. The cell below verifies each
rule against both `StubAgent` (Phase 1) and the already-implemented `VelocityAgent`
and `GeoRiskAgent` (Phase 2).

| Rule | Description |
|---|---|
| R1 | Inherits from `BaseAgent` |
| R2 | Accepts `config` dict and `logger` at `__init__` |
| R3 | Implements `fit(data) -> self` |
| R4 | Implements `predict(data) -> pd.DataFrame` with required columns |
| R5 | Implements `explain(transaction_id) -> str` |
| R6 | Serialisable (pickle / joblib) |
| R7 | Logs via structured logger (never `print()`) |
| R8 | Never mutates input data |
| R9 | Handles missing features gracefully |


In [ ]:
import inspect
import logging
import pickle
import pandas as pd
from src.agents.base_agent import BaseAgent
from src.agents.velocity_agent import VelocityAgent
from src.agents.geo_risk_agent import GeoRiskAgent


def _make_transactions() -> pd.DataFrame:
    return pd.DataFrame({
        'transaction_id': ['t1', 't2', 't3'],
        'timestamp': pd.to_datetime(
            ['2025-01-01 00:00', '2025-01-01 01:00', '2025-01-01 06:00'], utc=True
        ),
        'sender_account_id': ['ACC1', 'ACC1', 'ACC2'],
        'receiver_account_id': ['ACC2', 'ACC3', 'ACC1'],
        'amount_npr': [500_000.0, 900_000.0, 1_200_000.0],
        'sender_country': ['Nepal', 'Nepal', 'Qatar'],
        'receiver_country': ['Nepal', 'Nepal', 'Nepal'],
        'is_cross_border': [0, 0, 1],
        'remittance_corridor': [None, None, 'Qatar->Nepal'],
        'original_currency': ['NPR', 'NPR', 'QAR'],
        'ip_country': ['Nepal', 'Nepal', 'Qatar'],
        'ip_is_vpn': [0, 0, 1],
        'is_fraud': [0, 0, 1],
    })


txns = _make_transactions()
print('Sample transaction data shape:', txns.shape)


In [ ]:
def _check_compliance(AgentClass, txns: pd.DataFrame, name: str) -> None:
    results = {}

    # R1 — inherits BaseAgent
    results['R1 inherits BaseAgent'] = issubclass(AgentClass, BaseAgent)

    # R2 — accepts config dict and logger
    sig = inspect.signature(AgentClass.__init__)
    params = set(sig.parameters)
    results['R2 accepts config+logger'] = 'config' in params and 'logger' in params

    agent = AgentClass(config={}, logger=logging.getLogger(f'check.{name}'))

    # R3 — fit returns self
    ret = agent.fit(txns)
    results['R3 fit returns self'] = ret is agent

    # R4 — predict returns DataFrame with required columns
    pred = agent.predict(txns)
    required = {'transaction_id', 'risk_score', 'alert_flag', 'reason_code'}
    results['R4 predict columns'] = required.issubset(set(pred.columns))

    # R5 — explain returns str
    exp = agent.explain('t1')
    results['R5 explain returns str'] = isinstance(exp, str) and len(exp) > 0

    # R6 — pickle serialisable
    try:
        restored = pickle.loads(pickle.dumps(agent))
        results['R6 pickle serialisable'] = isinstance(restored, AgentClass)
    except Exception as exc:
        results['R6 pickle serialisable'] = f'FAIL: {exc}'

    # R7 — no print() calls in source
    src = inspect.getsource(AgentClass)
    results['R7 no print() in source'] = 'print(' not in src

    # R8 — input not mutated
    before = txns.copy(deep=True)
    agent.predict(txns)
    results['R8 input not mutated'] = txns.equals(before)

    # R9 — missing features handled gracefully
    minimal = pd.DataFrame({'transaction_id': ['t1']})
    try:
        res = agent.predict(minimal)
        results['R9 handles missing features'] = isinstance(res, pd.DataFrame)
    except Exception as exc:
        results['R9 handles missing features'] = f'FAIL: {exc}'

    print(f'\n=== {name} compliance ===')
    all_pass = True
    for rule, outcome in results.items():
        status = 'PASS' if outcome is True else ('FAIL' if outcome is False else f'NOTE: {outcome}')
        if outcome is not True:
            all_pass = False
        print(f'  [{status}] {rule}')
    print(f'  Overall: {"ALL PASS" if all_pass else "ISSUES DETECTED"}')


_check_compliance(StubAgent, txns, 'StubAgent')
_check_compliance(VelocityAgent, txns, 'VelocityAgent')
_check_compliance(GeoRiskAgent, txns, 'GeoRiskAgent')


---
## 7. Serialisation

AGENTS.md §8.1 rule 6: agents must be serialisable so trained agents can be saved and
loaded without refitting. `BaseAgent` handles logger exclusion transparently via
`__getstate__` / `__setstate__`, storing only the logger *name* so the module logger is
re-attached on deserialisation.

In [ ]:
import pickle
import joblib
import tempfile
from pathlib import Path

# ── pickle round-trip ─────────────────────────────────────────────────────────
stub = StubAgent(config={'stub_alert_threshold': 0.35})
stub.fit(sample_data)

blob = pickle.dumps(stub)
restored_pickle = pickle.loads(blob)

print('=== pickle round-trip ===')
print(f'  Type preserved:          {type(restored_pickle).__name__ == "StubAgent"}')
print(f'  alert_threshold:         {restored_pickle.alert_threshold}')
print(f'  is_fitted:               {restored_pickle.is_fitted}')
print(f'  logger name restored:    {restored_pickle.logger.name}')
print(f'  predict still works:     {len(restored_pickle.predict(sample_data)) == len(sample_data)}')


In [ ]:
# ── joblib round-trip (preferred for large numpy arrays inside models) ────────
with tempfile.TemporaryDirectory() as tmpdir:
    model_path = Path(tmpdir) / 'stub_agent.joblib'
    joblib.dump(stub, model_path)
    restored_joblib = joblib.load(model_path)

print('=== joblib round-trip ===')
print(f'  Type preserved:          {type(restored_joblib).__name__ == "StubAgent"}')
print(f'  alert_threshold:         {restored_joblib.alert_threshold}')
print(f'  is_fitted:               {restored_joblib.is_fitted}')
print(f'  logger name restored:    {restored_joblib.logger.name}')
print(f'  predict still works:     {len(restored_joblib.predict(sample_data)) == len(sample_data)}')


In [ ]:
# ── VelocityAgent fitted-model round-trip ────────────────────────────────────
from src.agents.velocity_agent import VelocityAgent

vel = VelocityAgent(config={'velocity_alert_threshold': 0.7})
vel.fit(txns)

vel_restored = pickle.loads(pickle.dumps(vel))
pred_before = vel.predict(txns)
pred_after  = vel_restored.predict(txns)

print('=== VelocityAgent pickle round-trip ===')
print(f'  Scores identical after restore: {(pred_before.risk_score.values == pred_after.risk_score.values).all()}')


---
## 8. Agent Registry Preview

The full Neural Sentinel system runs eight agents (AGENTS.md §8.2). Six are still to be
implemented in Phases 3–5. The table below documents the complete roster so Phase 1
design decisions account for every agent's needs.

| Phase | Agent | Class | Model | GPU? | Owner |
|---|---|---|---|---|---|
| 1 | Base interface | `BaseAgent` | Abstract | No | Dev 2 |
| 2 | Velocity | `VelocityAgent` | Isolation Forest | No | Dev 2 |
| 2 | Geo-Risk | `GeoRiskAgent` | CatBoost / LR | Optional | Dev 2 |
| 3 | Behaviour | `BehaviourAgent` | GRU / LSTM | Yes | Dev 2 |
| 3 | KYC/AML Rules | `KycAmlAgent` | Rule-based | No | Dev 2 |
| 4 | Graph | `GraphAgent` | GraphSAGE / GAT | Yes | Dev 2 |
| 4 | Meta-Learner | `MetaLearner` | Calibrated RF / XGBoost | Optional | Dev 2 |
| 1 (Data) | Data Quality | `DataQualityAgent` | Rule-based | No | Dev 1 |
| 5 | Explanation | `ExplanationAgent` | SHAP + templates | No | Dev 2 |


In [ ]:
# Preview: what the agent registry will look like when all agents are implemented
# This is design documentation — the imports will work after Phases 3-5 are complete.

PLANNED_AGENTS = [
    {'name': 'velocity',      'class': 'VelocityAgent',      'phase': 2, 'available': True},
    {'name': 'geo_risk',      'class': 'GeoRiskAgent',       'phase': 2, 'available': True},
    {'name': 'behaviour',     'class': 'BehaviourAgent',     'phase': 3, 'available': False},
    {'name': 'kyc_aml',       'class': 'KycAmlAgent',        'phase': 3, 'available': False},
    {'name': 'graph',         'class': 'GraphAgent',         'phase': 4, 'available': False},
    {'name': 'meta_learner',  'class': 'MetaLearner',        'phase': 4, 'available': False},
    {'name': 'explanation',   'class': 'ExplanationAgent',   'phase': 5, 'available': False},
]

print(f'  {"Agent":<20} {"Class":<25} {"Phase":<8} {"Status"}')
print('  ' + '-' * 65)
for entry in PLANNED_AGENTS:
    status = 'IMPLEMENTED' if entry['available'] else 'PLANNED'
    print(f"  {entry['name']:<20} {entry['class']:<25} Phase {entry['phase']:<3} {status}")


In [ ]:
# Demonstrate that already-implemented agents load from the package
from src.agents import BaseAgent, VelocityAgent, GeoRiskAgent

registry = {
    'velocity': VelocityAgent,
    'geo_risk': GeoRiskAgent,
}

print('Currently registered agents:')
for name, cls in registry.items():
    a = cls()
    print(f'  {name:<15} agent_name={a.agent_name!r:<20} threshold={a.alert_threshold}')


### 8.1 Meta-learner input contract

The Meta-Learner (Phase 4) joins all agent outputs on `transaction_id`.  The join
produces one row per transaction with `N` risk-score columns where `N` is the number
of agents.  This is only possible if every agent:

- uses the same `transaction_id` values (guaranteed by `build_predictions`).
- names its output column `risk_score` (guaranteed by `prediction_columns`).
- produces exactly one row per input transaction (checked by the compliance test above).

In [ ]:
# Simulate the meta-learner join with the two Phase 2 agents
from src.utils.config import get_config

cfg = get_config()
vel   = VelocityAgent(cfg).fit(txns)
geo   = GeoRiskAgent(cfg).fit(txns)

v_pred = vel.predict(txns)[['transaction_id', 'risk_score', 'alert_flag', 'reason_code']].rename(
    columns={'risk_score': 'velocity_score', 'alert_flag': 'velocity_flag', 'reason_code': 'velocity_reason'}
)
g_pred = geo.predict(txns)[['transaction_id', 'risk_score', 'alert_flag', 'reason_code']].rename(
    columns={'risk_score': 'geo_score', 'alert_flag': 'geo_flag', 'reason_code': 'geo_reason'}
)

meta_input = txns[['transaction_id', 'is_fraud']].merge(v_pred, on='transaction_id').merge(g_pred, on='transaction_id')
print('Meta-learner input preview (Phase 4 target schema):')
print(meta_input.to_string(index=False))


---
## 9. Inline Test-Suite Dry Run

The Phase 1 tests from `tests/test_agents.py` are replicated here so the notebook
is self-contained and can be run end-to-end on Kaggle without a local `pytest`
installation.

Each assertion corresponds directly to a test function in the test file. A summary
table is printed at the end.

In [ ]:
import traceback

_RESULTS: list[tuple[str, bool, str]] = []


def _test(name: str):
    """Decorator that runs a zero-arg test function and records the result."""
    def _wrap(fn):
        try:
            fn()
            _RESULTS.append((name, True, ''))
            print(f'  [PASS] {name}')
        except Exception:
            _RESULTS.append((name, False, traceback.format_exc(limit=2)))
            print(f'  [FAIL] {name}')
        return fn
    return _wrap


print('Running Phase 1 test battery...')
print()


In [ ]:
@_test('test_agent_initializes_with_agent_specific_threshold')
def _t1():
    agent = StubAgent(config={'stub_alert_threshold': 0.7})
    assert agent.alert_threshold == 0.7
    assert agent.agent_name == 'stub'
    assert isinstance(agent.logger, logging.Logger)


@_test('test_predict_returns_complete_canonical_output_without_mutating_input')
def _t2():
    data = pd.DataFrame({'transaction_id': ['txn-1', 'txn-2'], 'amount_npr': [10.0, 20.0]})
    original = data.copy(deep=True)
    result = StubAgent(config={'alert_threshold': 0.1}).fit(data).predict(data)
    assert list(result.columns) == list(BaseAgent.prediction_columns)
    assert result['alert_flag'].tolist() == [1, 1]
    pd.testing.assert_frame_equal(data, original)


@_test('test_missing_transaction_id_returns_schema_safe_empty_output')
def _t3():
    result = StubAgent().predict(pd.DataFrame({'amount_npr': [100.0]}))
    assert result.empty
    assert list(result.columns) == list(BaseAgent.prediction_columns)


@_test('test_explain_returns_human_readable_text')
def _t4():
    explanation = StubAgent().explain('txn-1')
    assert 'txn-1' in explanation


@_test('test_agent_is_pickle_serializable')
def _t5():
    import pickle as _pickle
    restored = _pickle.loads(_pickle.dumps(StubAgent(config={'alert_threshold': 0.8})))
    assert isinstance(restored, StubAgent)
    assert restored.alert_threshold == 0.8
    assert restored.logger.name == 'neural_sentinel.agents.stub'


@_test('test_invalid_threshold_raises_value_error')
def _t6():
    try:
        StubAgent(config={'alert_threshold': 1.5})
        raise AssertionError('Expected ValueError not raised')
    except ValueError:
        pass  # expected


In [ ]:
# VelocityAgent tests
from src.agents.velocity_agent import VelocityAgent
import pytest


@_test('test_velocity_agent_returns_schema_and_does_not_mutate')
def _t7():
    data = _make_transactions()
    original = data.copy(deep=True)
    result = VelocityAgent(config={'velocity_alert_threshold': 0.5}).fit(data).predict(data)
    assert list(result.columns) == list(BaseAgent.prediction_columns)
    assert len(result) == len(data)
    assert result.risk_score.between(0, 1).all()
    pd.testing.assert_frame_equal(data, original)


@_test('test_velocity_agent_handles_missing_features')
def _t8():
    minimal = pd.DataFrame({'transaction_id': ['txn-1']})
    result = VelocityAgent().fit(minimal).predict(minimal)
    assert len(result) == 1
    assert result.risk_score.iloc[0] == 0.0


@_test('test_velocity_agent_raises_if_predict_called_before_fit')
def _t9():
    data = _make_transactions()
    try:
        VelocityAgent().predict(data)
        raise AssertionError('Expected RuntimeError not raised')
    except RuntimeError as exc:
        assert 'must be fitted' in str(exc)


In [ ]:
# GeoRiskAgent tests
from src.agents.geo_risk_agent import GeoRiskAgent


@_test('test_geo_risk_agent_detects_vpn_and_cross_border')
def _t10():
    data = _make_transactions()
    result = GeoRiskAgent().fit(data).predict(data)
    assert list(result.columns) == list(BaseAgent.prediction_columns)
    vpn_reason = result.loc[result.transaction_id == 't3', 'reason_code'].item()
    assert vpn_reason == 'VPN_OR_PROXY'
    assert result.risk_score.between(0, 1).all()


@_test('test_geo_risk_agent_handles_missing_features')
def _t11():
    result = GeoRiskAgent().predict(pd.DataFrame({'transaction_id': ['txn-1']}))
    assert len(result) == 1
    assert result.risk_score.iloc[0] == 0.0


# Summary
print()
passed = sum(1 for _, ok, _ in _RESULTS if ok)
failed = len(_RESULTS) - passed
print(f'Results: {passed}/{len(_RESULTS)} passed, {failed} failed')
if failed:
    print('\nFailed tests:')
    for name, ok, tb in _RESULTS:
        if not ok:
            print(f'  {name}:')
            print(tb)


---
## 10. Phase 1 Summary

### What was designed and validated in Phase 1

| Artefact | File | Status |
|---|---|---|
| `BaseAgent` abstract class | `src/agents/base_agent.py` | Complete |
| `Transaction` Pydantic model | `src/data_contracts.py` | Complete |
| `Account` Pydantic model | `src/data_contracts.py` | Complete |
| `AlertScore` Pydantic model | `src/data_contracts.py` | Complete |
| `Config` singleton | `src/utils/config.py` | Complete |
| Nepal-context constants | `src/utils/nepal_context.py` | Complete |
| `StubAgent` test template | `tests/test_agents.py` | Complete |

### Key design decisions made in Phase 1

1. **Single output schema** — `prediction_columns` is a class-level tuple on `BaseAgent`;
   every agent's `predict()` return value is validated against it by `build_predictions`.

2. **Optional-dependency pattern** — CatBoost, IsolationForest, and GNN libraries are
   guarded by `try/except ImportError` at module level; deterministic fallbacks ensure
   CPU-only Kaggle free-tier runs never crash.

3. **Config as single source of truth** — all thresholds and weights live in
   `Config`; agents never hardcode numbers.

4. **Serialisation contract** — `__getstate__`/`__setstate__` strip the live logger
   and re-attach it by name on deserialisation; all other state is preserved for
   joblib model storage.

5. **Compliance checklist** — nine rules from AGENTS.md §8.1 are encoded as automated
   checks; future agents can run the same `_check_compliance()` helper.

### Handoff to Phase 2

Phase 2 (`notebooks/detection/phase2_velocity_geo_agents.ipynb`) builds on this
foundation by implementing:
- `VelocityAgent` — Isolation Forest on rolling account-level features.
- `GeoRiskAgent` — CatBoost on geographic and corridor signals.

Both agents are already implemented and passing the compliance checklist above.

### Open questions for later phases

- **Behaviour agent sequence length**: What is the optimal window of past transactions
  to feed into the GRU encoder? (Phase 3)
- **Graph batching strategy**: Which `torch_geometric` data loader to use for the
  5M-row transaction graph without OOM on Kaggle free tier? (Phase 4)
- **Meta-learner calibration**: Platt scaling vs. isotonic regression; to be decided
  after evaluating meta-learner reliability diagram on held-out data. (Phase 4)

In [ ]:
# Final confirmation: all imports and contracts are healthy
from src.agents import BaseAgent, VelocityAgent, GeoRiskAgent
from src.data_contracts import Transaction, Account, AlertScore
from src.utils.config import get_config
from src.utils.nepal_context import NRB_CASH_REPORTING_THRESHOLD_NPR

print('All Phase 1 imports healthy')
print(f'NRB reporting threshold: NPR {NRB_CASH_REPORTING_THRESHOLD_NPR:,.0f}')
print(f'Agents available in src.agents: {[c.__name__ for c in [BaseAgent, VelocityAgent, GeoRiskAgent]]}')
print('Phase 1 notebook complete.')
